# 🖼️ Path 3: Image-to-Class (Direct Visual Diagnosis)

**Research Context**: This path treats ECG analysis as a computer vision task. By transforming signals into images, we can leverage state-of-the-art CNN architectures to detect morphological anomalies.

### 🌟 Supervisor's Guide:
1.  **Shape Recognition**: See how the model distinguishes between different 'shapes' of heartbeats.
2.  **Activation Heatmaps**: Visualizing the neural filters that detect specific curves and spikes.
3.  **End-to-End Learning**: Demonstrating the model's ability to classify directly from pixels without manual feature engineering.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import os
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

## 📊 1. Loading Real MIT-BIH Data
We load the signals and their clinical labels (0: Normal, 1: Supraventricular, 2: Ventricular, 3: Fusion, 4: Unknown).

In [ ]:
DATA_PATH = "sample_mitbih.csv"
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH, header=None)
else:
    print("⚠️ Warning: sample_mitbih.csv not found. Using dummy data.")
    df = pd.DataFrame(np.random.randn(100, 188))
    df.iloc[:, 187] = np.random.randint(0, 5, 100)

signals = df.iloc[:, :187].values
labels = df.iloc[:, 187].values.astype(int)

class_names = ["Normal", "SVEB", "VEB", "Fusion", "Unknown/Paced"]
print(f"✅ Loaded {len(signals)} heartbeats with labels.")

## 🖼️ 2. Labeled Image Generation
We create a visual dataset where each heartbeat is an image.

In [ ]:
def signal_to_image(signal, img_size=(128, 128)):
    fig, ax = plt.subplots(figsize=(1.5, 1.5), dpi=64)
    ax.plot(signal, color='black', linewidth=2)
    ax.axis('off')
    fig.canvas.draw()
    
    w, h = fig.canvas.get_width_height()
    img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    img = img.reshape((h, w, 3))
    plt.close(fig)
    
    img_gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    img_resized = cv2.resize(img_gray, img_size)
    return img_resized / 255.0

print("🎨 Generating visual dataset (this may take a few seconds)...")
X_imgs = []
y_labels = []

for i in tqdm(range(min(200, len(signals)))):
    X_imgs.append(signal_to_image(signals[i]))
    y_labels.append(labels[i])

X_imgs = np.array(X_imgs)
y_labels = np.array(y_labels)

# Show samples from different classes
plt.figure(figsize=(15, 4))
for i in range(5):
    plt.subplot(1, 5, i+1)
    indices = np.where(y_labels == i)[0]
    if len(indices) > 0:
        plt.imshow(X_imgs[indices[0]], cmap='gray')
        plt.title(f"{class_names[i]}")
    plt.axis('off')
plt.suptitle("ECG Images for Different Diagnostic Classes", fontsize=16)
plt.show()

## 🧠 3. Image Classifier Model
A 2D CNN that learns to distinguish the shape of a normal beat from an arrhythmia.

In [ ]:
class ImageClassifier(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = ImageClassifier().to(device)
X_tensor = torch.tensor(X_imgs, dtype=torch.float32).unsqueeze(1).to(device)
y_tensor = torch.tensor(y_labels, dtype=torch.long).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002)

## 📈 4. Training loop
The model adjusts its weights to minimize classification error.

In [ ]:
print("🎓 Training end-to-end image classifier...")
for epoch in range(15):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_tensor)
    loss = criterion(outputs, y_tensor)
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 3 == 0:
        _, preds = torch.max(outputs, 1)
        acc = (preds == y_tensor).float().mean()
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Accuracy: {acc.item():.2%}")

print("✨ Model is ready for demonstration.")

### 🏥 Note on 'Unknown' Class (Category Q):
The **Unknown** class is a standard clinical category in the MIT-BIH dataset. It includes:
- **Paced beats** (from artificial pacemakers)
- **Unclassifiable signals** (distorted by noise or artifacts)

In a real-world deployment, 'Unknown' triggers a **manual review** by a cardiologist, ensuring AI safety.

## 🔍 5. Internal Mechanism: Feature Activation
How does the model distinguish between classes? It looks for edges, curvature, and relative peak positions.

In [ ]:
def show_comparative_activations(model, img_tensor, indices):
    model.eval()
    fig, axes = plt.subplots(len(indices), 6, figsize=(18, 3*len(indices)), facecolor='#f8f9fa')
    
    for idx_row, sample_idx in enumerate(indices):
        activation = {}
        def get_activation(name):
            def hook(model, input, output):
                activation[name] = output.detach().cpu().numpy()
            return hook
        handle = model.features[0].register_forward_hook(get_activation('conv1'))
        model(img_tensor[sample_idx:sample_idx+1])
        handle.remove()
        act = activation['conv1'][0]
        axes[idx_row, 0].imshow(img_tensor[sample_idx, 0].cpu(), cmap='gray')
        axes[idx_row, 0].set_title(f"Input: {class_names[y_labels[sample_idx]]}", fontsize=10, fontweight='bold')
        axes[idx_row, 0].axis('off')
        for i in range(5):
            axes[idx_row, i+1].imshow(act[i], cmap='inferno')
            axes[idx_row, i+1].set_title(f"Filter {i+1}", fontsize=9)
            axes[idx_row, i+1].axis('off')
    plt.suptitle("Comparative Neural Response: How the AI Sees Different Heartbeat Morphologies", fontsize=18, y=1.02)
    plt.tight_layout()
    plt.show()
n_idx = np.where(y_labels == 0)[0][0]
v_idx = np.where(y_labels == 2)[0][0]
show_comparative_activations(model, X_tensor, [n_idx, v_idx])